In [3]:
import os
os.chdir(r'C:\Users\Ibraheem khan\Downloads\Ml_LAB\LAB 10')

In [4]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
import os
from tensorflow.keras.callbacks import Callback

In [5]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

In [6]:
def MLP():
    model = Sequential()
    model.add(Flatten(input_shape=(time_steps , num_features)))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))
    return model

In [7]:
model1 = MLP()
model1.summary()

C:\Users\Ibraheem khan\anaconda3\envs\tf_env\lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 504)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │        16,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,193 (63.25 KB)

 Trainable params: 16,193 (63.25 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) for `plot_model` to work.


In [9]:
checkpoints = r'C:\Users\Ibraheem khan\Downloads\Ml_LAB\LAB 10\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'C:\Users\Ibraheem khan\Downloads\Ml_LAB\LAB 10'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

In [10]:
os.path.exists(JSON_PATH)

False

In [11]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor = TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1]

In [12]:
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =MLP()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


In [13]:
import os
path_dataset =r'C:\Users\Ibraheem khan\Downloads\Ml_LAB\LAB 10'
path_tr = os.path.join(path_dataset, 'train.csv')
df_tr = pd.read_csv(path_tr)
train_set = df_tr.iloc[:].values
path_v = os.path.join(path_dataset, 'validation.csv')
df_v = pd.read_csv(path_v)
validation_set = df_v.iloc[:].values 
path_te = os.path.join(path_dataset, 'test.csv')
df_te = pd.read_csv(path_te)
test_set = df_te.iloc[:].values 

path_scaler = os.path.join(path_dataset, 'AEP_scaler.pkl')
scaler         = pickle.load(open(path_scaler, 'rb'))

train_set.shape, validation_set.shape, test_set.shape

C:\Users\Ibraheem khan\anaconda3\envs\tf_env\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


((860, 21), (90, 21), (30, 21))

In [14]:
start = time.time()
train_X , train_y = univariate_multi_step(train_set, time_steps, target_col=0,target_len=1)
validation_X, validation_y = univariate_multi_step(validation_set, time_steps, target_col=0,target_len=1)
test_X, test_y = univariate_multi_step(test_set, time_steps, target_col=0,target_len=1)
print('Time Consumed', time.time()-start, "sec")

Time Consumed 0.006072521209716797 sec


In [15]:
train_X.shape

(835, 24, 21)

In [16]:
epochs = 23
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                    verbose = verbose)

Epoch 1/23
22/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2366 - mae: 0.2366 - mape: 129.2670   
Epoch 1: val_loss improved from None to 0.09505, saving model to C:\Users\Ibraheem khan\Downloads\Ml_LAB\LAB 10\E1-cp-0001-loss0.10.h5


27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.1622 - mae: 0.1622 - mape: 92.7867 - val_loss: 0.0950 - val_mae: 0.0950 - val_mape: 33.8962
Epoch 2/23
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 0.1124 - mae: 0.1124 - mape: 50.6930
Epoch 2: val_loss improved from 0.09505 to 0.08748, saving model to C:\Users\Ibraheem khan\Downloads\Ml_LAB\LAB 10\E1-cp-0002-loss0.09.h5


27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0808 - mae: 0.0808 - mape: 48.2781 - val_loss: 0.0875 - val_mae: 0.0875 - val_mape: 28.7825
Epoch 3/23
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0809 - mae: 0.0809 - mape: 41.2328 
Epoch 3: val_loss improved from 0.08748 to 0.07290, saving model to C:\Users\Ibraheem khan\Downloads\Ml_LAB\LAB 10\E1-cp-0003-loss0.07.h5


27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0778 - mae: 0.0778 - mape: 38.0365 - val_loss: 0.0729 - val_mae: 0.0729 - val_mape: 25.7779
Epoch 4/23
24/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0841 - mae: 0.0841 - mape: 40.1923 
Epoch 4: val_loss did not improve from 0.07290
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0793 - mae: 0.0793 - mape: 42.9066 - val_loss: 0.1860 - val_mae: 0.1860 - val_mape: 71.3779
Epoch 5/23
22/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1041 - mae: 0.1041 - mape: 56.8054 
Epoch 5: val_loss improved from 0.07290 to 0.05591, saving model to C:\Users\Ibraheem khan\Downloads\Ml_LAB\LAB 10\E1-cp-0005-loss0.06.h5


27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0815 - mae: 0.0815 - mape: 45.0605 - val_loss: 0.0559 - val_mae: 0.0559 - val_mape: 18.2527
Epoch 6/23
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 0.0579 - mae: 0.0579 - mape: 31.2999
Epoch 6: val_loss did not improve from 0.05591
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0713 - mae: 0.0713 - mape: 38.3120 - val_loss: 0.0711 - val_mae: 0.0711 - val_mape: 25.8491
Epoch 7/23
26/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0497 - mae: 0.0497 - mape: 22.9265 
Epoch 7: val_loss improved from 0.05591 to 0.05284, saving model to C:\Users\Ibraheem khan\Downloads\Ml_LAB\LAB 10\E1-cp-0007-loss0.05.h5


27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0498 - mae: 0.0498 - mape: 27.0092 - val_loss: 0.0528 - val_mae: 0.0528 - val_mape: 17.7318
Epoch 8/23
25/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0686 - mae: 0.0686 - mape: 32.0182 
Epoch 8: val_loss did not improve from 0.05284
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0711 - mae: 0.0711 - mape: 36.9024 - val_loss: 0.1044 - val_mae: 0.1044 - val_mape: 39.0914
Epoch 9/23
25/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0601 - mae: 0.0601 - mape: 34.1633 
Epoch 9: val_loss did not improve from 0.05284
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0520 - mae: 0.0520 - mape: 27.9842 - val_loss: 0.0741 - val_mae: 0.0741 - val_mape: 27.2869
Epoch 10/23
23/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0521 - mae: 0.0521 - mape: 31.3987  
Epoch 10: val_loss did not improve from 0.05284
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0582 - mae: 0.0582 - mape: 31.1729 - val_loss: 0.0780 - val_mae: 0.0780 - val_mape: 27.8254
Epoc

27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0510 - mae: 0.0510 - mape: 26.6541 - val_loss: 0.0436 - val_mae: 0.0436 - val_mape: 14.7705
Epoch 12/23
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0524 - mae: 0.0524 - mape: 30.4554 
Epoch 12: val_loss did not improve from 0.04360
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0471 - mae: 0.0471 - mape: 24.6163 - val_loss: 0.0454 - val_mae: 0.0454 - val_mape: 15.1746
Epoch 13/23
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 0.0520 - mae: 0.0520 - mape: 18.2144
Epoch 13: val_loss did not improve from 0.04360
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0508 - mae: 0.0508 - mape: 23.4855 - val_loss: 0.0574 - val_mae: 0.0574 - val_mape: 20.4381
Epoch 14/23
23/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0568 - mae: 0.0568 - mape: 25.8522  
Epoch 14: val_loss did not improve from 0.04360
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0480 - mae: 0.0480 - mape: 24.3350 - val_loss: 0.0495 - val_mae: 0.0495 - val_mape: 19.4617


27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0356 - mae: 0.0356 - mape: 16.3448 - val_loss: 0.0426 - val_mae: 0.0426 - val_mape: 13.5526
Epoch 21/23
 1/27 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - loss: 0.0359 - mae: 0.0359 - mape: 10.7322
Epoch 21: val_loss did not improve from 0.04257
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0430 - mae: 0.0430 - mape: 21.1845 - val_loss: 0.0505 - val_mae: 0.0505 - val_mape: 16.3666
Epoch 22/23
21/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0518 - mae: 0.0518 - mape: 34.3849 
Epoch 22: val_loss did not improve from 0.04257
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0476 - mae: 0.0476 - mape: 27.4951 - val_loss: 0.0468 - val_mae: 0.0468 - val_mape: 16.5710
Epoch 23/23
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0432 - mae: 0.0432 - mape: 20.1749 
Epoch 23: val_loss improved from 0.04257 to 0.03717, saving model to C:\Users\Ibraheem khan\Downloads\Ml_LAB\LAB 10\E1-cp-0023-loss0.04.h5


27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0423 - mae: 0.0423 - mape: 21.5642 - val_loss: 0.0372 - val_mae: 0.0372 - val_mape: 14.0629


In [17]:

model = load_model(r'C:\Users\Ibraheem khan\Downloads\Ml_LAB\LAB 10\E1-cp-0023-loss0.04.h5',compile=False)

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
Mean Absolute Error (MAE): 972.57
Median Absolute Error (MedAE): 1026.21
Mean Squared Error (MSE): 1213047.29
Root Mean Squared Error (RMSE): 1101.38
Mean Absolute Percentage Error (MAPE): 6.19 %
Median Absolute Percentage Error (MDAPE): 6.62 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)


# Fine Tuning

In [18]:
checkpoints = r'C:\Users\Ibraheem khan\Downloads\Ml_LAB\LAB 10\E1-cp-0023-loss0.04.h5'
model=r'C:\Users\Ibraheem khan\Downloads\Ml_LAB\LAB 10\E1-cp-0023-loss0.04.h5'
start_epoch= 24

In [20]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

TypeError: 'TrainingMonitor' object is not callable

In [19]:
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
print(TrainingMonitor)
print(type(TrainingMonitor))

<class 'timeseires.callbacks.TrainingMonitor.TrainingMonitor'>
<class 'type'>


In [2]:
print(type(TrainingMonitor))

<class 'type'>


In [22]:
epochs = 10
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Train on 835 samples, validate on 65 samples
Epoch 1/10
672/835 [=======================>......] - ETA: 0s - loss: 0.0220 - mae: 0.0220 - mape: 9.3523 
Epoch 00001: val_loss improved from inf to 0.03299, saving model to C:\Users\arif\Downloads\delete\E2-cp-0001-loss0.03.h5
835/835 [==============================] - 1s 623us/sample - loss: 0.0213 - mae: 0.0213 - mape: 9.0533 - val_loss: 0.0330 - val_mae: 0.0330 - val_mape: 9.9388
Epoch 2/10
384/835 [============>.................] - ETA: 0s - loss: 0.0182 - mae: 0.0182 - mape: 7.2160
Epoch 00002: val_loss improved from 0.03299 to 0.03287, saving model to C:\Users\arif\Downloads\delete\E2-cp-0002-loss0.03.h5
835/835 [==============================] - 0s 239us/sample - loss: 0.0176 - mae: 0.0176 - mape: 7.8068 - val_loss: 0.0329 - val_mae: 0.0329 - val_mape: 10.0569
Epoch 3/10
576/835 [===================>..........] - ETA: 0s - loss: 0.0170 - mae: 0.0170 - mape: 6.8712
Epoch 00003: val_loss improved from 0.03287 to 0.02768, saving model 

In [23]:

model = load_model(r'C:\Users\arif\Downloads\delete\E1-cp-0055-loss0.03.h5')

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

Mean Absolute Error (MAE): 12401.71
Median Absolute Error (MedAE): 12529.37
Mean Squared Error (MSE): 153967995.29
Root Mean Squared Error (RMSE): 12408.38
Mean Absolute Percentage Error (MAPE): 79.25 %
Median Absolute Percentage Error (MDAPE): 79.69 %


y_test_unscaled.shape=  (5, 1)
y_pred.shape=  (5, 1)
